# Lasy losowe i gradient boosting #
## Wprowadzenie

Na poprzednich zajęciach pracowaliśmy z modelem **drzewa decyzyjnego**, analizując jego działanie na zbiorze danych *breast_cancer*.

W tym notebooku rozszerzymy tę wiedzę, przechodząc do dwóch bardzo ważnych rodzin modeli zespołowych (ang. *ensemble methods*):

- **Lasy losowe** (*Random Forest*, metoda bagging)
- **Gradient Boosting** (boosting drzew decyzyjnych)

Modele te bazują na drzewach decyzyjnych jako klasyfikatorach bazowych, ale różnią się sposobem, w jaki wykorzystują wiele drzew, aby poprawić jakość predykcji.

### 🔹 Bagging — Lasy losowe

**Random Forest** wykorzystuje koncepcję *baggingu* (bootstrap aggregating):
- trenuje **wiele niezależnych drzew decyzyjnych**,
- każde drzewo dostaje losową próbkę danych (z powtórzeniami),
- oraz losowy podzbiór cech przy podziale węzłów,
- a predykcja końcowa to **głosowanie większościowe** (lub średnia).

### 🔹 Boosting — Gradient Boosting

**Gradient Boosting** działa inaczej:
- drzewa budowane są **sekwencyjnie**, jedno po drugim,
- każde kolejne drzewo stara się **poprawić błędy poprzednich**,
- model uczy się „reszt błędu”, czyli tego, czego nie wyjaśniły wcześniejsze drzewa.

# Przygotowanie środowiska programistycznego

In [42]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier

import matplotlib.pyplot as plt
import plotly.graph_objects as go
import seaborn as sns
from termcolor import colored
from sklearn.metrics import roc_auc_score

# Załadowanie danych — **Pima Indians Diabetes Dataset**

W tym ćwiczeniu będziemy pracować ze zbiorem danych dotyczącym diagnozowania cukrzycy u kobiet z populacji Pima Indian. Jest to klasyczny, trudny zbiór używany w uczeniu maszynowym do testowania modeli klasyfikacyjnych.

Każdy wiersz reprezentuje jedną pacjentkę, a kolumny zawierają różne cechy kliniczne i demograficzne, które mogą mieć wpływ na wystąpienie cukrzycy typu 2. Celem jest przewidzenie zmiennej label, która informuje, czy u danej osoby wykryto cukrzycę (`1`) lub nie (`0`).

## Opis cech:

- **Pregnancies**  
  Liczba przebytych ciąż. Może być wskaźnikiem pewnych zmian metabolicznych lub hormonalnych.

- **Glucose**  
  Stężenie glukozy w osoczu po 2 godzinach od testu obciążenia glukozą.  
  **Najważniejsza cecha** w diagnozowaniu cukrzycy.

- **BloodPressure**  
  Ciśnienie tętnicze rozkurczowe (mm Hg).  
  Zbyt niskie lub zbyt wysokie wartości mogą wskazywać na problemy metaboliczne.

- **SkinThickness**  
  Grubość fałdu skórnego tricepsa (mm).  
  Koreluje z poziomem tkanki tłuszczowej.

- **Insulin**  
  Stężenie insuliny w surowicy po 2 godzinach (μU/ml).  
  Wskazuje na insulinooporność lub problemy z gospodarką glukozową.

- **BMI**  
  Wskaźnik masy ciała (*Body Mass Index*).  
  Wysokie BMI jest jednym z głównych czynników ryzyka cukrzycy.

- **DiabetesPedigreeFunction**  
  Wskaźnik określający skłonność genetyczną do cukrzycy na podstawie historii rodzinnej.  
  Wyższa wartość → większe ryzyko.

- **Age**  
  Wiek pacjentki.  
  Ryzyko cukrzycy zwiększa się wraz z wiekiem.

- **label**  
  Zmienna docelowa:  
  - `1` — cukrzyca wykryta  
  - `0` — brak diagnozy


W kolejnych zadaniach wykorzystamy ten zbiór danych do treningu i porównania modeli z rodziny drzew decyzyjnych.

In [ ]:
url = "https://www.fuw.edu.pl/~mpoziomska/data/diabetes2.csv"
df = pd.read_csv(url, encoding='latin-1')
print(df.head())

**Proszę:**
* podzielić dane na części uczącą i walidacyjną w proporcjach 80:20

**Uwaga**: można użyć parametru ```random_state=42``` by umożliwić porównanie wyników z innymi osobami

**Wskazówka**: proszę skorzystać z funkcji ```sklearn.model_selection.train_test_split```

In [ ]:
#BEGIN_SOLUTION
df_train, df_test = train_test_split(df, test_size = 0.2, random_state=4456782, stratify=df['label'])
print(colored("Train dataset:\n","blue"),df_train["label"].describe())
print(colored("Test dataset:\n","blue"),df_test["label"].describe())
#END_SOLUTION
pass

## Zadanie
Eksperyment z parametrami Random Forest — analiza AUC

W tym zadaniu przeprowadzimy bardziej zaawansowaną analizę klasyfikatora **RandomForestClassifier**, badając, jak parametry modelu wpływają na jego skuteczność.

**Proszę:**
1. Zainicjalizować i wytrenować modele **RandomForestClassifier** dla wielu kombinacji parametrów:
   - `n_estimators: [1, 5, 10, 50, 100]` - Określa liczbę drzew tworzonych w lesie
   - `max_depth: [1, 2, 4, 6, None]` - Maksymalna głębokość każdego drzewa.

2. Dla każdej kombinacji:
   - wytrenować model,
   - obliczyć **AUC**,
   - zapisać wynik.

3. Utworzyć **heatmapę**, która pokaże zależność AUC od `n_estimators` i `max_depth`.

---

### Wskazówka

Do obliczenia AUC możesz użyć:
```python
from sklearn.metrics import roc_auc_score
roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])


In [ ]:
#BEGIN_SOLUTION
X_train = df_train.drop("label", axis=1)
y_train = df_train["label"]
X_test = df_test.drop("label", axis=1)
y_test = df_test["label"]

n_estimators_list = [1, 5, 10, 50, 100]
max_depth_list = [1, 2, 4, 6, None]

results = []

for n in n_estimators_list:
    for depth in max_depth_list:
        # 1. Inicjalizacja modelu
        model = RandomForestClassifier(
            n_estimators=n,
            max_depth=depth,
            random_state=42
        )
        
        # 2. Trening
        model.fit(X_train, y_train)
        
        # 3. Predykcja i obliczanie AUC
        auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
        
        # 4. Zapis wyników
        results.append({
            "n_estimators": n,
            "max_depth": depth if depth is not None else "None",
            "AUC": auc
        })

# Konwersja do DataFrame
df_rf_results = pd.DataFrame(results)
df_results_pivot = df_rf_results.pivot(
    index="max_depth",
    columns="n_estimators",
    values="AUC"
)

# Heatmapa
plt.figure(figsize=(8, 6))
sns.heatmap(df_results_pivot, annot=True, fmt=".3f", cmap="YlGnBu")
plt.title("AUC w zależności od n_estimators i max_depth")
plt.show()
#END_SOLUTION

## Zadanie  
### Eksperyment z parametrami Gradient Boosting — analiza AUC

Teraz powtórzymy analogiczny eksperyment jak w przypadku Random Forest, ale użyjemy modelu **GradientBoostingClassifier**.

Gradient Boosting łączy wiele płytkich drzew decyzyjnych tworzonych sekwencyjnie — każde kolejne drzewo stara się poprawić błędy poprzednich.

**Proszę:**

1. Przetestować model **GradientBoostingClassifier** dla wielu kombinacji parametrów:
   - `n_estimators: [10, 50, 100, 200]` — liczba drzew w sekwencji boostingowej  
   - `learning_rate: [0.01, 0.05, 0.1, 0.2]` — szybkość uczenia (mniejsza → model uczy się wolniej, ale stabilniej)

2. Dla każdej kombinacji parametrów:
   - wytrenować model,
   - obliczyć AUC na zbiorze testowym,
   - zapisać wynik do tabeli.

3. Utworzyć **heatmapę**, która pokaże wpływ `n_estimators` i `learning_rate` na AUC.


In [ ]:
#BEGIN_SOLUTION
n_estimators_list = [1, 5, 10, 50, 100, 200]
learning_rates = [0.001, 0.01, 0.05, 0.1, 0.2]

results = []

for n in n_estimators_list:
    for lr in learning_rates:
        model = GradientBoostingClassifier(
            n_estimators=n,
            learning_rate=lr,
            random_state=42
        )
        model.fit(X_train, y_train)
        y_pred_proba = model.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, y_pred_proba)
        
        results.append({
            "n_estimators": n,
            "learning_rate": lr,
            "AUC": auc
        })

df_gb_results = pd.DataFrame(results)
pivot = df_gb_results.pivot(
    index="learning_rate",
    columns="n_estimators",
    values="AUC"
)

plt.figure(figsize=(8, 5))
sns.heatmap(pivot, annot=True, fmt=".3f", cmap="YlGnBu")
plt.title("Gradient Boosting — AUC dla różnych parametrów")
plt.ylabel("learning_rate")
plt.xlabel("n_estimators")
plt.show()
#END_SOLUTION

# Zadanie: Porównanie modeli drzewiastych — wybór najlepszych parametrów i walidacja krzyżowa

W tym zadaniu porównamy trzy modele oparte na drzewach decyzyjnych:

1. **Drzewo decyzyjne** 
2. **Random Forest**
3. **Gradient Boosting** 

---
**Proszę:**
- Wybrać po jednym zestawie najlepszych parametrów dla RF i GB.
- Z poprzednich zajęć odszukać najlepsze parametry dla **Drzewa Decyzyjnego**
- Zainicjalizować model z **wybranymi najlepszymi parametrami**
- Przeprowadzić **5-krotną walidację krzyżową** (`cross_val_score`) na całym zbiorze danych
- Jako metrykę wybrać AUC
- Wypisać średnią i odchylenie standardowe dla trzech modeli


In [ ]:
#BEGIN_SOLUTION
# --- Random Forest z najlepszymi parametrami ---
best_rf = df_rf_results.loc[df_rf_results["AUC"].idxmax()]
best_rf

rf_model = RandomForestClassifier(
    n_estimators=int(best_rf['n_estimators']),
    max_depth=None if best_rf['max_depth'] == "None" else int(best_rf['max_depth']),
    random_state=42
)
cv_auc_score_rf = cross_val_score(rf_model, df.drop(columns=['label']), df.label, cv=5, scoring='roc_auc')

# --- Gradient Boosting z najlepszymi parametrami ---
best_gb = df_gb_results.loc[df_gb_results["AUC"].idxmax()]
best_gb

gb_model = GradientBoostingClassifier(
    n_estimators=int(best_gb['n_estimators']),
    learning_rate=float(best_gb['learning_rate']),
    random_state=42
)
cv_auc_score_gb = cross_val_score(gb_model, df.drop(columns=['label']), df.label, cv=5, scoring='roc_auc')

# --- Decision tree classifier z najlepszymi parametrami ---

dtc_model = DecisionTreeClassifier(random_state=42, ccp_alpha=0.01)
cv_auc_score_dtc = cross_val_score(dtc_model, df.drop(columns=['label']), df.label, cv=5, scoring='roc_auc')

# --- Podsumowanie wyników ---
print(f"Decision Tree Classifier CV AUC: {np.mean(cv_auc_score_dtc):.4f} ± {np.std(cv_auc_score_dtc):.4f}")
print(f"Random Forest Classifier CV AUC: {np.mean(cv_auc_score_rf):.4f} ± {np.std(cv_auc_score_rf):.4f}")
print(f"Gradient Boosting Classifier CV AUC: {np.mean(cv_auc_score_gb):.4f} ± {np.std(cv_auc_score_gb):.4f}")

#END_SOLUTION


## Zadanie  
### Analiza ważności cech (Feature Importance) w Random Forest i Gradient Boosting

Modele drzewiaste mają tę zaletę, że potrafią oszacować **ważność cech** — czyli informują, które atrybuty najbardziej wpływają na decyzję modelu.

W tym zadaniu porównamy ważność cech między:
- najlepszym modelem **Random Forest** (z poprzedniego zadania),
- najlepszym modelem **Gradient Boosting** (wybranym z poprzedniego ćwiczenia).

**Proszę:**

1. Wytrenować:
   - model Random Forest z optymalnymi parametrami (wybrany wcześniej),
   - model Gradient Boosting z parametrami, które dały najwyższe AUC.

2. Wyciągnąć wektory `feature_importances_` dla każdego modelu.

3. Stworzyć wykresy:
   - osobny wykres słupkowy dla Random Forest,
   - osobny wykres słupkowy dla Gradient Boosting,
   - dodatkowo: **jeden wspólny wykres**, który pokazuje różnice obok siebie  
     (np. porównanie RF vs GB dla każdej cechy).

4. Odpowiedzieć na pytania:
   - Które cechy mają najwyższe znaczenie w obu modelach?
   - Czy modele zgadzają się co do rankingu cech?
   - Czy któryś model wydaje się bardziej „skoncentrowany” na mniejszej liczbie cech?

**Wskazówka:**  
Wykres słupkowy możesz stworzyć np. tak:

```python
importances = model.feature_importances_
plt.barh(features, importances)


In [ ]:
#BEGIN_SOLUTION
rf_model.fit(X_train, y_train)
gb_model.fit(X_train, y_train)

rf_importances = rf_model.feature_importances_
gb_importances = gb_model.feature_importances_
features = X_train.columns

df_imp = pd.DataFrame({
    "feature": features,
    "RF": rf_importances,
    "GB": gb_importances
}).set_index("feature")


fig = go.Figure()

fig.add_trace(go.Bar(
    y=df_imp.index,
    x=df_imp["RF"],
    orientation='h',
    name='Random Forest',
    opacity=0.7
))

fig.add_trace(go.Bar(
    y=df_imp.index,
    x=df_imp["GB"],
    orientation='h',
    name='Gradient Boosting',
    opacity=0.7
))

fig.update_layout(
    title="Porównanie feature importance: Random Forest vs Gradient Boosting",
    xaxis_title="Importance",
    yaxis_title="Feature",
    barmode='group',
    height=600
)

fig.show()
#END_SOLUTION

# Zadanie domowe — Klasyfikacja na zbiorze **Titanic** z użyciem **CatBoost**

## Przypomnienie

Zbiór **Titanic** to klasyczny dataset używany w uczeniu maszynowym. Zawiera informacje o pasażerach słynnego statku, takie jak:
- wiek,
- płeć,
- klasa biletu,
- liczba członków rodziny na pokładzie,
- cena biletu,
- port zaokrętowania,

oraz zmienną docelową **Survived**, która określa, czy dany pasażer przeżył katastrofę.

---

## Wprowadzenie — model **CatBoost**

**CatBoost** (Categorical Boosting) to nowoczesny model gradient boosting, który:
- **natywnie obsługuje dane kategoryczne**,
- jest odporny na overfitting,
- dobrze działa nawet przy niewielkiej liczbie próbek,
- często przewyższa jakością modele takie jak Random Forest czy klasyczny Gradient Boosting, gdy zbiór zawiera wiele zmiennych kategorycznych.

W tym zadaniu użyjemy CatBoostClassifier.

---

## Zadanie
 
1. Zidentyfikuj kolumny **kategoryczne**.
3. Wybierz cechy, które według Ciebie nadają się do predykcji czy dany pasażer przeżyje.
4. Usuń wiersze, dla których są brakujące wartości.
2. Zainicjalizuj model:
   - użyj `CatBoostClassifier`,
   - wyłącz wypisywanie logów (`verbose=False`),
3. Przeprowadź **5-foldową walidację krzyżową** z metryką **AUC**.
4. Wypisz średnią oraz odchylenie standardowe AUC.
5. Narysuj historie uczenia modelu dla każdego z foldów.

---

### Wskazówki

- Użyj:
  ```python
  from catboost import CatBoostClassifier
    

In [ ]:

from sklearn.datasets import fetch_openml
data = fetch_openml(name='titanic', version=1, as_frame=True)
df = data.frame
df.head()

In [ ]:
#BEGIN_SOLUTION
from sklearn.model_selection import KFold, cross_val_score
from catboost import CatBoostClassifier

# Wybór cech
features = ['pclass', 'sex', 'age', 'fare', 'embarked', 'sibsp', 'parch']
df = df[features + ['survived']].copy()

# Usuwanie wierszy z brakującymi danymi
df = df.dropna().sample(frac=1).reset_index(drop=True)

# Definicja kolumn kategorycznych
cat_cols = ['pclass', 'sex', 'embarked']

# Przygotowanie danych
X = df[features]
y = df['survived']

# Walidacja krzyżowa
kf = KFold(n_splits=5, shuffle=True, random_state=42)

auc_scores = []
eval_histories = []

for train_idx, test_idx in kf.split(X):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    
    model = CatBoostClassifier(
        iterations=700,
        learning_rate=0.05,
        depth=6,
        verbose=False,
        cat_features=cat_cols,
        eval_metric='AUC'
    )
    
    model.fit(X_train, y_train, eval_set=(X_test, y_test), use_best_model=True)
    eval_histories.append(model.get_evals_result())

    preds = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, preds)
    auc_scores.append(auc)

print("\nCatBoost AUC scores:", auc_scores)
print("CatBoost AUC mean:", round(pd.Series(auc_scores).mean(), 4))
print("CatBoost AUC std:", round(pd.Series(auc_scores).std(), 4))

# Wizualizacja historii uczenia:
fig = go.Figure()
for i, history in enumerate(eval_histories):
    fig.add_trace(go.Scatter(
        y=history['validation']['AUC'],
        mode='lines',
        name=f'Fold {i+1}'
    ))
fig.update_layout(
    title='CatBoost AUC w kolejnych iteracjach (walidacja krzyżowa)',
    xaxis_title='Iteracje',
    yaxis_title='AUC',
    height=500
)
fig.show()
#END_SOLUTION